Have to check that the cells that we're interested in are not place cells

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

# JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# "JAL3_7sept", "JAL3_4sept", "JAL3_1sept", "JAL3_25aug", "JAL3_22aug",

experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar, # (unmatched number of neurons and cluster ids) # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_14may, JAL8_flip4_10may]

In [30]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.visualize.efizz.heatmap import assign_positional_bins_to_frames, single_unit_heatmap_plotting, robust_min_max_calculation
from behave_analysis.utils.heatplot_utils import filter_outside_arena_tracking_for_video_and_spike_data, add_features_binned
from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe
from JR_test_scripts.escape.escape_utils import load_homing

import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd
from loguru import logger

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
"""Overhead for the whole notebook"""
case = 'either_tuned' # 'escape_tuned', 'dist_tuned', 'either_tuned
condition = 'all' # 'shelter_only', 'barrier', 'flipped_barrier', 'all', 'barrier_both'
c_names = ['shelter_only', 'barrier', 'flipped_barrier']
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]

for exp in experiments_objects:
    nickname = exp.nick_name + '_' + exp.experiment_date
    print(nickname)
    
    cells = extract_significant_cells(exp, case, condition)

    vdf_with_bins, tracking_data, unit_ids, clu_names = load_data(exp, cells)

    plot_heatmaps(vdf_with_bins, unit_ids, clu_names, conditions, exp, nickname)

JAL004_2023_09_03


2025-03-07 12:18:04.425 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL004_2023_09_19


2025-03-07 12:35:11.439 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL004_2023_08_28


...

2025-03-07 12:53:43.391 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL005_2023_09_08


2025-03-07 13:09:14.483 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL005_2023_09_21


2025-03-07 13:15:48.115 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


In [93]:
"""How many cells are tuned only to the %escape and not to the distance to shelter in exploration?
Using residuals to identify tuning to %escape surviving from distance to shelter subtraction"""

def extract_significant_cells(exp, case, condition):

    # 1. load in explore tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
    saving_file = dump_path + exp_nickname + '_Tuning.npz'
    data = np.load(saving_file)

    # pull out the params
    exp_params_real = data['params_real']
    exp_params_shifts = data['params_shifts']

    # identify cells that are sig tuned to distance to shelter in exploration
    exp_sig_dist = exp_params_real[:,:,0] > np.nanpercentile(exp_params_shifts[:,:,:,0], 95, axis = 0)

    # 2. load in escape homing/escape tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    saving_file = dump_path + exp_nickname + '_ProperTuning.npz'
    data = np.load(saving_file)

    fr_esc = data['fr_real']
    params_esc = data['params_real']
    params_esc_shift = data['params_shifts']

    # identify cells that are sig tuned to %escape in homing/escape
    sig_escape = params_esc[:,:,0] > np.nanpercentile(params_esc_shift[:,:,:,0], 95, axis = 0)

    # 2. load in dist to shelter homing/escape tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    saving_file = dump_path + exp_nickname + '_ProperTuning.npz'
    data = np.load(saving_file)

    fr_dist = data['fr_real']
    params_dist = data['params_real']
    params_dist_shifts = data['params_shifts']

    # identify cells that are sig tuned to %escape in homing/escape
    sig_dist = params_dist[:,:,0] > np.nanpercentile(params_dist_shifts[:,:,:,0], 95, axis = 0)

    # 4. load in residuals data
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    saving_file = dump_path + exp_nickname + '_ResidualsTuning.npz'
    data = np.load(saving_file)
    params_real_exp = data['params_real_exp']
    params_shifts_exp_res = data['params_shifts_exp_res']

    # find cells whose residual tuning to %escape - distance to shelter in exploration is significant
    sig_res = params_real_exp[:,:,0] > np.nanpercentile(params_shifts_exp_res[:,:,:,0], 95, axis = 0)

    """Select the cells I want to analyse"""
    if case == 'escape_tuned':
        # cells that are tuned to %escape in homing/escape (subselect ones that are not tuned to distance to shelter in exploration and passed the residual test)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_escape[:, c] == True) & (exp_sig_dist[:, c] == False) & (sig_res[:, c] == False)  # V1 only
            AC = (sig_escape[:, c] == True) & (sig_res[:, c] == True)  # Both V1 and V1 regressed
            xval[:,c] = (A == True) | (AC == True)

    if case == 'dist_tuned':
        # cells that are tuned to distance to shelter in homing/escape (subselect ones that are not tuned to %escape in homing/escape)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_dist[:, c] == True)
            B = (sig_dist[:, c] == True) & (exp_sig_dist[:,c] == False)
            xval[:,c] = (A == True) # xval[:,c] = (B == True)

    if case == 'either_tuned':
        # cells that are tuned to distance to shelter or %escape in homing/escape (subselect ones that are not tuned to distance to shelter in exploration and passed the residual test)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_escape[:, c] == True) & (exp_sig_dist[:, c] == False) & (sig_res[:, c] == False)
            AC = (sig_escape[:, c] == True) & (sig_res[:, c] == True) # Both V1 and V1 regressed
            B = (sig_dist[:, c] == True)
            xval[:,c] = (A == True) | (AC == True) | (B == True)

    if condition == 'all':
        cells = np.sum(xval, axis = 1) == 3
    elif condition == 'shelter_only':
        cells = xval[:,0] == True
    elif condition == 'barrier':
        cells = xval[:,1] == True
    elif condition == 'flipped_barrier':
        cells = xval[:,2] == True
    elif condition == 'barrier_both':
        cells = (xval[:,0] == False) & (xval[:,1] == True) & (xval[:,2] == True)
    
    return cells

In [94]:
"""Load data"""
def load_data(exp, cells):
    # load session
    session = Process(exp).load_session()

    # load video & spike data
    COLUMNS_TO_KEEP = [
        "mouse_x_position",
        "mouse_y_position",
        "spike_clusters",
        "spike_count",
        "OutofshelterIdx",
        "EscapePeriod",
        "shelter",
        "barrier_present",
        "barrier_flipped",
        "homingPeriod", # 'homingPeriod'
    ]
    base_path = os.path.join(session.base_path, session.processed_path)
    video_and_spike_data = pl.read_parquet(
                    os.path.join(base_path + "\\" + "good_video_spike_count_df.parquet"), 
                    low_memory=True,
                    use_pyarrow = True,
                    memory_map=True,
                )


    # load homings
    _, _, homing_bool = load_homing(session, int(np.amax(np.unique(video_and_spike_data['frames'].to_numpy()))))
    video_and_spike_data = video_and_spike_data.with_columns(
        pl.col('frames').cast(pl.Int64).alias('frames')
    )
    homing_frames = [homing_bool[frame-1] for frame in video_and_spike_data['frames'].to_list()]
    video_and_spike_data = video_and_spike_data.hstack([pl.Series("homingPeriod", homing_frames)])

    # post process video and spike data
    video_and_spike_data = video_and_spike_data.select(COLUMNS_TO_KEEP)
    clean_video_df = filter_outside_arena_tracking_for_video_and_spike_data(video_and_spike_data=video_and_spike_data, session=session).to_pandas()
    vdf_with_bins, x_bin_nums, y_bin_nums = assign_positional_bins_to_frames(video_df=clean_video_df, nbins=30)

    # tracking data
    tracking_data = open_tracking_data(session)

    # select units of interest
    unit_ids = video_and_spike_data["spike_clusters"].unique().to_numpy()
    if unit_ids[0] == 0: unit_ids = unit_ids[1:]
    unit_ids = unit_ids[cells]
    clu_names = np.where(cells == True)[0]

    return vdf_with_bins, tracking_data, unit_ids, clu_names

In [95]:
def plot_heatmaps(vdf_with_bins, unit_ids, clu_names, conditions, exp, nickname):    
    # Loop through each unit and create a heatmap for each condition

    for n_clu, clu in enumerate(unit_ids):
        total_spikes = 0
        fig, axs = plt.subplots(nrows=1, ncols=len(conditions), figsize=(15, 7), sharey=True, sharex=True)
        cbar_ax = fig.add_axes([0.91, 0.3, 0.02, 0.4])  # The list represents [left, bottom, width, height]

        min_spikes = 0
        max_spikes = 0

        for idx, condition in enumerate(conditions):
            pldf = pl.DataFrame(vdf_with_bins)
            filtered_df = filter_video_dataframe(dataframe=pldf, condition=condition, exclude_escape=True, exclude_homings=True)
            behave_pivot = filtered_df.to_pandas().groupby(["x_bins", "y_bins"]).size().reset_index(name="total_entries")
            behave_pivot = behave_pivot.pivot(index="y_bins", columns="x_bins", values="total_entries").fillna(-1)
            clu_df = filtered_df.filter(pl.col("spike_clusters") == clu)

            # Skip if no data for this unit and condition
            if clu_df.is_empty() or sum(clu_df["spike_count"]) == 0:
                logger.warning(f"Unit {clu} has no data for condition {condition}, skipping...")
                continue

            pddf = clu_df.to_pandas()

            # Count and normalise the spikes in each bin
            spike_counts_in_bin = pddf.groupby(["x_bins", "y_bins"])["spike_count"].sum().reset_index()
            total_entries_in_bins = pddf.groupby(["x_bins", "y_bins"]).size().reset_index(name="total_entries")  # Total entries in each bin
            normalized_df = pd.merge(spike_counts_in_bin, total_entries_in_bins, on=["x_bins", "y_bins"])
            normalized_df["normalized_spike_count"] = (
                spike_counts_in_bin["spike_count"] / total_entries_in_bins["total_entries"]
            ) * 40  # spikes per second
            assert sum(total_entries_in_bins["total_entries"]) <= len(
                pddf
            ), "Total entries in bins is greater than the initial dataframe, can not be!"

            condition_spikes = sum(spike_counts_in_bin["spike_count"])

            # Y bins are the rows, x bins are the columns and the values are firing rates
            pivot_df = normalized_df.pivot(index="y_bins", columns="x_bins", values="normalized_spike_count").fillna(0)
            pivot_df = pivot_df.mask(behave_pivot == -1, -1)  # Mask the -1 values (no data / exploration) with set_bad color

            min_spikes, max_spikes = robust_min_max_calculation(pivot_df, min_spikes, max_spikes)

            # Create the heatmap
            # add_features_binned(axs[idx], condition, tracking_data, x_bin_nums, y_bin_nums)
            single_unit_heatmap_plotting(
                axs=axs,
                idx=idx,
                heatmap_data=pivot_df,
                cbar_ax=cbar_ax,
                condition=condition,
                condition_spikes=condition_spikes,
                min_spikes=min_spikes,
                max_spikes=max_spikes,
            )

        fig.suptitle(f"Unit {clu_names[n_clu]}: heatmap - Total spikes across conditions {total_spikes:.1f}", fontsize=20)
        file_name = f"neuron{clu_names[n_clu]}_place_heatmap.png"
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'place_heatmap'
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)
        fig.savefig(dump_path + "\\" + file_name)
        plt.clf()
        plt.close("all")